> **Parameterized notebook** — merged from the AWQ original and its
> `_unquantized` bf16 twin (both preserved verbatim in `archive/premerge/`,
> executed run records in `archive/executed/` and `outputs/`).
> Set `QUANTIZATION` in the parameters cell below (`"awq"` | `"bf16"`);
> bf16 outputs carry the historical `_unquantized` suffix in `resources/`.
> Headless: `make run_notebook N=<this notebook> P=bf16`.


# 21 — E3: negative / mixed steering

One **attracted** + one **repelled** concept, on the three E0 ρ strata, both
directions per pair, at the E1 operating point (k=4, steps=24, shared 20-prompt
suite and baselines):

| method | space | composition |
|---|---|---|
| `attract-only` | — | vanilla CGD on the attract concept (control: measures *incidental* expression of the repelled concept) |
| `frame-diff` | frame | guide = attract − repel (the paper's differential guidance; identical to `Concept.average(weights=[1,−1])` — Step 5) |
| `score-neg` | score | per-step z-scored weighted sum with weights [1, −1] |

Unnormalized score-space `[1, −1]` is provably identical to `frame-diff`
(linearity confound, Step 4), so it is not run.

**Success here is two-sided:** Δexpr(attract) > 0 AND Δexpr(repel) < 0.
The quantities of interest are suppression (−Δrepel) beyond what attract-only
already buys incidentally, and whether frame- or score-space buys it cheaper
in fluency. The high-ρ stratum (near-synonyms sorrow/sadness) is the hard case:
suppressing a concept while expressing its geometric twin.

In [ ]:
QUANTIZATION = "awq"  # "awq" (historical default) | "bf16" (unquantized)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation

In [ ]:
CONFIGS = {
    "awq": (
        "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
        "cuda:0",
        torch.float16,
    ),
    "bf16": ("meta-llama/Llama-3.1-8B-Instruct", "auto", torch.bfloat16),
}
MODEL_ID, DEVICE_MAP, TORCH_DTYPE = CONFIGS[QUANTIZATION]
SUFFIX = "" if QUANTIZATION == "awq" else "_unquantized"

fur = FrameUnembeddingRepresentation.from_model_id(
    MODEL_ID,
    device_map=DEVICE_MAP,
    torch_dtype=TORCH_DTYPE,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24

SELECTION = json.loads(Path("resources/15_e0_selected_pairs.json").read_text())[
    "selected"
]
for stratum, info in SELECTION.items():
    print(f"{stratum:>6}: {info['a']} / {info['b']}  rho={info['rho']:.3f}")

In [ ]:
def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


# Same suite as 20_e1_family_sweep — keeps E1/E3 rows directly comparable.
PROMPTS = [
    chat("What men can be?", "1."),
    chat("What women can be?", "1."),
    chat("Tell me a short story.", "Once"),
    chat("Describe your favorite place.", "My favorite place is"),
    chat("What do you like to do on weekends?", "I like"),
    chat("Give me some advice for a good life.", "1."),
    chat(
        "What is the most important thing in the world?", "The most important thing is"
    ),
    chat("Describe the weather today.", "Today the weather is"),
    chat("What should I cook for dinner?", "You should"),
    chat("Name things that make people happy.", "1."),
    chat("What did you do today?", "Today I"),
    chat("Describe a memorable meal.", "The most memorable meal"),
    chat("What is a good gift idea?", "A good gift"),
    chat("Describe your hometown.", "My hometown is"),
    chat("Tell me about your favorite season.", "My favorite season is"),
    chat("What makes a good friend?", "A good friend"),
    chat("Describe a place you would like to visit.", "I would like to visit"),
    chat("What do you do to relax?", "To relax, I"),
    chat("Tell me something interesting.", "Here is something interesting:"),
    chat("Describe your morning routine.", "My morning starts"),
]

LOG_PATH = Path(f"resources/22_e3_negative{SUFFIX}.jsonl")
LOG_PATH.unlink(missing_ok=True)

harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(PROMPTS, max_new_tokens=STEPS + 2)
print("baseline ready")

In [ ]:
FAMILY = {
    "attract-only": "control",
    "frame-diff": "frame",
    "score-neg": "score",
}


def make_methods(attract, repel) -> dict:
    differential = attract - repel  # Procrustes projection of the difference
    return {
        "attract-only": lambda: fur.generate_with_topk_guide(
            PROMPTS, guide=attract, k=K, steps=STEPS
        ),
        "frame-diff": lambda: fur.generate_with_topk_guide(
            PROMPTS, guide=differential, k=K, steps=STEPS
        ),
        "score-neg": lambda: fur.generate_with_topk_multi_guide(
            PROMPTS,
            guides=[attract, repel],
            weights=[1.0, -1.0],
            k=K,
            steps=STEPS,
            normalize="zscore",
        ),
    }

In [ ]:
for stratum, info in SELECTION.items():
    concept = {
        name: fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
        for name in (info["a"], info["b"])
    }

    for attract_name, repel_name in ((info["a"], info["b"]), (info["b"], info["a"])):
        methods = make_methods(concept[attract_name], concept[repel_name])
        for method, generate in methods.items():
            torch.cuda.empty_cache()
            texts, probe = generate()
            harness.evaluate(
                PROMPTS,
                texts,
                [attract_name, repel_name],
                config={
                    "stratum": stratum,
                    "rho": info["rho"],
                    "method": method,
                    "family": FAMILY[method],
                    "attract": attract_name,
                    "repel": repel_name,
                    "k": K,
                    "steps": STEPS,
                },
                probe=probe,
                baseline_texts=baseline_texts,
                concepts=concept,
            )
            print(f"{stratum} / +{attract_name} -{repel_name} / {method}: done")

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert len(parsed) == 3 * 2 * len(FAMILY) * len(PROMPTS), "3 strata x 2 dir x methods"
assert all(r["expression"] is not None for r in parsed), "metric v2 present everywhere"
print(f"gate: {len(parsed)} records parse, expression logged")

In [ ]:
rows = []
for record in parsed:
    config = record["config"]
    expr = record["expression"]
    delta_attract = expr[config["attract"]]["delta"]
    delta_repel = expr[config["repel"]]["delta"]
    rows.append(
        {
            "stratum": config["stratum"],
            "rho": config["rho"],
            "method": config["method"],
            "attract": config["attract"],
            "repel": config["repel"],
            "direction": f"+{config['attract'].split('.')[0]}"
            f"-{config['repel'].split('.')[0]}",
            "delta_attract": delta_attract,
            "delta_repel": delta_repel,
            "net": delta_attract - delta_repel,
            "clean": delta_attract > 0 and delta_repel < 0,
            "ppl_ratio": record["ppl_ratio"],
            "fluency_flag": record["fluency_flag"],
        }
    )
results = pd.DataFrame(rows)

summary = (
    results.groupby(["stratum", "rho", "direction", "method"])
    .agg(
        delta_attract=("delta_attract", "mean"),
        delta_repel=("delta_repel", "mean"),
        net=("net", "mean"),
        clean_rate=("clean", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        flag_share=("fluency_flag", "mean"),
    )
    .reset_index()
    .sort_values(["rho", "direction", "net"], ascending=[True, True, False])
)
summary.to_csv(f"resources/22_e3_summary{SUFFIX}.csv", index=False)
summary.round(4)

## Quadrant view

x = Δexpr(repelled), y = Δexpr(attracted); each point is one (direction, method)
cell mean. The target quadrant is **top-left** (attract up, repel below
baseline). `attract-only` shows where steering lands with no repulsion at all —
the other two methods must beat it leftward without giving up height.

In [ ]:
method_colors = {
    "attract-only": "tab:gray",
    "frame-diff": "tab:blue",
    "score-neg": "tab:orange",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharex=True, sharey=True)
for ax, (stratum, info) in zip(axes, SELECTION.items()):
    cell = summary[summary["stratum"] == stratum]
    for _, row in cell.iterrows():
        marker = (
            "o" if row["direction"].startswith(f"+{info['a'].split('.')[0]}") else "s"
        )
        ax.scatter(
            row["delta_repel"],
            row["delta_attract"],
            s=80,
            marker=marker,
            color=method_colors[row["method"]],
            edgecolor="black",
            linewidth=0.5,
        )
        ax.annotate(
            f"{row['method']}\n{row['direction']}",
            (row["delta_repel"], row["delta_attract"]),
            textcoords="offset points",
            xytext=(6, 2),
            fontsize=6.5,
        )
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(0, color="gray", lw=0.8)
    ax.set_title(f"{stratum} (rho={info['rho']:.2f})")
    ax.set_xlabel("mean delta expr (repelled)  <- better")
axes[0].set_ylabel("mean delta expr (attracted)  better ->")

handles = [
    plt.Line2D([], [], marker="o", ls="", color=c, label=m)
    for m, c in method_colors.items()
]
axes[-1].legend(handles=handles, fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(f"resources/22_e3_quadrants{SUFFIX}.png", dpi=150)
plt.show()

In [ ]:
# Paired per-prompt suppression: does either negative method push the repelled
# concept below attract-only's incidental level, on the same prompt?
paired = results.pivot_table(
    index=["stratum", "direction", "attract", "repel"],
    columns="method",
    values="delta_repel",
    aggfunc=list,
)
print("suppression vs attract-only control (paired per prompt):")
for method in ("frame-diff", "score-neg"):
    for (stratum, direction, _, _), row in paired.iterrows():
        control, treated = pd.Series(row["attract-only"]), pd.Series(row[method])
        wins = int((treated < control).sum())
        print(
            f"  {stratum:>6} {direction:>18} {method:>10}: "
            f"{wins:>2}/{len(treated)} prompts more suppressed, "
            f"mean gap {(treated - control).mean():+.3f}"
        )

In [ ]:
def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]


for stratum in SELECTION:
    for method in FAMILY:
        example = next(
            r
            for r in parsed
            if r["config"]["stratum"] == stratum
            and r["config"]["method"] == method
            and "short story" in r["prompt"]
        )
        config = example["config"]
        deltas = {
            name.split(".")[0]: round(v["delta"], 3)
            for name, v in example["expression"].items()
        }
        print(
            f"[{stratum} / +{config['attract'].split('.')[0]}"
            f"-{config['repel'].split('.')[0]} / {method}] deltas={deltas}"
        )
        print("   ", answer(example["text"])[:130])
    print()